<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/query_transformations/StepBackQueryTransformDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step-Back Query Transform

If you're opening this Notebook on colab, you will probably need to install LlamaIndex.

In [ ]:
!pip install llama-index

### Download Data

In [ ]:
!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'

#### Load documents and build the VectorStoreIndex

In [ ]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.indices.query.query_transform.base import (
    StepBackQueryTransform,
)
from llama_index.core.query_engine import TransformQueryEngine
from IPython.display import Markdown, display

# load documents
documents = SimpleDirectoryReader("./data/paul_graham/").load_data()
index = VectorStoreIndex.from_documents(documents)

## Example: Step-Back Query Transform improves retrieval for specific questions

Step-back prompting (Zheng et al., 2023) abstracts a specific question into a higher-level, principle-oriented question before retrieval. This helps when the original query contains narrow identifiers, dates, or entities that may not appear verbatim in the retrieved documents.

In [ ]:
query_str = "What did Paul Graham do after going to RISD?"

#### First, query *without* transformation

In [ ]:
query_engine = index.as_query_engine()
response = query_engine.query(query_str)
display(Markdown(f"<b>{response}</b>"))

#### Now, apply `StepBackQueryTransform`

The transform generates a step-back question — removing the specific names and dates and asking about the underlying pattern. This abstracted question is used for embedding lookup.

In [ ]:
step_back = StepBackQueryTransform()
step_back_query_engine = TransformQueryEngine(query_engine, step_back)

# Inspect the step-back question that gets generated
step_back_bundle = step_back(query_str)
print(f"Original query:  {query_str}")
print(f"Step-back query: {step_back_bundle.query_str}")

In [ ]:
response = step_back_query_engine.query(query_str)
display(Markdown(f"<b>{response}</b>"))

## Example: Step-Back helps with entity-specific queries

When a query mentions a very specific entity or date that may not appear verbatim in the source text, step-back abstraction can improve recall.

In [ ]:
query_str = "What school did Alice attend between Aug and Nov 1954?"

#### Querying without transformation

In [ ]:
response = query_engine.query(query_str)
display(Markdown(f"<b>{response}</b>"))

#### Querying with StepBack

In [ ]:
step_back_bundle = step_back(query_str)
print(f"Original query:  {query_str}")
print(f"Step-back query: {step_back_bundle.query_str}")

response = step_back_query_engine.query(query_str)
display(Markdown(f"<b>{response}</b>"))

## Customising the step-back prompt

You can provide your own prompt using `build_step_back_prompt`.

In [ ]:
from llama_index.core.indices.query.query_transform.base import (
    StepBackQueryTransform,
    build_step_back_prompt,
)
from llama_index.core.prompts.prompt_type import PromptType

custom_prompt = build_step_back_prompt(
    system_instructions=(
        "You are an expert at abstracting questions about historical events. "
        "Rewrite questions to ask about the general historical pattern or process, "
        "removing specific names and dates."
    ),
    few_shot_examples=[
        (
            "Who signed contract #4711 on 2024-03-15?",
            "What is the contract-signing process?",
        ),
        (
            "How many Grand Slam titles does the winner of the 2020 Australian Open have?",
            "What determines a tennis player's Grand Slam success?",
        ),
    ],
    prompt_type=PromptType.STEP_BACK,
)

step_back_custom = StepBackQueryTransform(step_back_prompt=custom_prompt)
query_str = "What did Paul Graham do after RISD?"
step_back_bundle = step_back_custom(query_str)
print(f"Custom step-back: {step_back_bundle.query_str}")